In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

tfidf = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.5, stop_words='english')
X_tfidf = tfidf.fit_transform(newsgroups.data)

print(f"원본 TF-IDF 행렬 차원: {X_tfidf.shape}")

원본 TF-IDF 행렬 차원: (2034, 1000)


In [3]:
from sklearn.decomposition import TruncatedSVD
# 10개의 주성분(토픽)으로 축소
svd = TruncatedSVD(n_components=10, random_state=42)
x_svd = svd.fit_transform(X_tfidf)
x_svd.shape, svd.explained_variance_ratio_.sum()

((2034, 10), np.float64(0.07138973486263994))

In [8]:
# 잠재의미(topic)별 핵심단어 확인
import numpy as np
terms = tfidf.get_feature_names_out()
for i, comp in enumerate(svd.components_):
    # 각 컴포넌트에서 가중치가 높은 상위 10개 단어 추출
    top_terms_index = np.argsort(-comp)[:10]
    print(top_terms_index)
    top_terms = [terms[idx] for idx in top_terms_index]
    print(f"topic {i+1}: {', '.join(top_terms)}")


[275 639 461 372 897 833 496 469 272 376]
topic 1: don, people, just, god, think, space, like, know, does, good
[372 452 639 126 123 775 171 733 275 897]
topic 2: god, jesus, people, bible, believe, say, christian, religion, don, think
[833 578 481 806 569 618 855 519 283 215]
topic 3: space, nasa, launch, shuttle, moon, orbit, station, lunar, earth, cost
[372 833 452 287 126 769 272 893 123 104]
topic 4: god, space, jesus, edu, bible, satan, does, thanks, believe, atheism
[893 272 469 833  76  53 602 522 571 511]
topic 5: thanks, does, know, space, anybody, advance, objective, mail, morality, looking
[602 571 570 944 420 272 337 933 230 780]
topic 6: objective, morality, moral, values, image, does, file, use, data, science
[897 833 287 275 461 338 372 578 380 337]
topic 7: think, space, edu, don, just, files, god, nasa, graphics, file
[338 337 350 275 469 687 420 377 372 902]
topic 8: files, file, format, don, know, program, image, got, god, tiff
[372 185 461 768 134 113 856 571 602 9

In [11]:
# 유사도기반 검색 테스트 (차원 축소 효과)
# 단어가 직접 겹치지 않아도 의미적으로 유사한 문서를 찾을 수 있는지 확인

from sklearn.metrics.pairwise import cosine_similarity
sim_scores = cosine_similarity(x_svd[0:1], x_svd)
top_index = np.argsort(-sim_scores)[:5]
print(f"가장 유사한 문서 top5 index : {top_index}")
print(f"유사도 점수 : {sim_scores[0][top_index]}")

가장 유사한 문서 top5 index : [[   0 1892 1209 ...  226   81 2030]]
유사도 점수 : [[ 1.          0.98994107  0.98315076 ... -0.21622939 -0.24119456
  -0.24399704]]


In [12]:
newsgroups.data[0], newsgroups.target_names[ newsgroups.target[0] ]

("Hi,\n\nI've noticed that if you only save a model (with all your mapping planes\npositioned carefully) to a .3DS file that when you reload it after restarting\n3DS, they are given a default position and orientation.  But if you save\nto a .PRJ file their positions/orientation are preserved.  Does anyone\nknow why this information is not stored in the .3DS file?  Nothing is\nexplicitly said in the manual about saving texture rules in the .PRJ file. \nI'd like to be able to read the texture rule information, does anyone have \nthe format for the .PRJ file?\n\nIs the .CEL file format available from somewhere?\n\nRych",
 'comp.graphics')

In [14]:
newsgroups.target_names

['alt.atheism', 'comp.graphics', 'sci.space', 'talk.religion.misc']